# Retrieval playground

Rulează pipeline-ul de retrieval (query rewrite -> hybrid search semantic + BM25 -> fusion -> rerank -> top-K) pe întrebări ad-hoc, și inspectează rezultatele la fiecare etapă.

Presupune că `uv run python ingest.py` a fost deja rulat (există `db/`).

In [2]:
from dotenv import load_dotenv

load_dotenv(override=True)

# from config import FINAL_K, RETRIEVAL_K
from app.rag_simple.retrieval.pipeline import retrieve_with_trace

FINAL_K = 20
RETRIEVAL_K = 20

print(f"RETRIEVAL_K={RETRIEVAL_K}  FINAL_K={FINAL_K}")


RETRIEVAL_K=20  FINAL_K=20


## Întrebări de test

Editează lista de mai jos cu întrebările pe care vrei să le testezi ad-hoc.

In [3]:
QUESTIONS = [
    "cum se tratează cancerul cu macrobiotica?",
    "ce este echilibrul Yin-Yang?",
    "ce rol au organele in psihologia macrobiotica?",
]


## Rulare + inspecție pe etape

In [4]:
def show_chunk(i, doc, show_context=False):
    source = doc.metadata.get("source", "?")
    heading = doc.metadata.get("h4") or doc.metadata.get("h3") or ""
    text = doc.page_content if show_context else doc.metadata.get("original_text", doc.page_content)
    preview = text.strip().replace("\n", " ")
    if len(preview) > 220:
        preview = preview[:220] + "..."
    print(f"[{i:>2}] {source} | {heading}\n     {preview}\n")


def show_trace(trace, top_n=10):
    print("=" * 100)
    print(f"QUESTION:   {trace.question}")
    print(f"REWRITTEN:  {trace.rewritten_question}")
    print(f"counts -> semantic={len(trace.semantic_hits)}  lexical={len(trace.lexical_hits)}  "
          f"fused={len(trace.fused)}  reranked={len(trace.reranked)}  final={len(trace.final)}")

    print("\n--- semantic hits (top 5) ---")
    for i, doc in enumerate(trace.semantic_hits[:5], start=1):
        show_chunk(i, doc)

    print("--- lexical / BM25 hits (top 5) ---")
    for i, doc in enumerate(trace.lexical_hits[:5], start=1):
        show_chunk(i, doc)

    print(f"--- final, reranked, top {top_n} (of {len(trace.final)}) ---")
    for i, doc in enumerate(trace.final[:top_n], start=1):
        show_chunk(i, doc)


In [5]:
traces = {}
for question in QUESTIONS:
    traces[question] = retrieve_with_trace(question)
    show_trace(traces[question])


QUESTION:   cum se tratează cancerul cu macrobiotica?
REWRITTEN:  tratamentul cancerului prin dieta macrobiotica
counts -> semantic=20  lexical=28  fused=29  reranked=29  final=20

--- semantic hits (top 5) ---
[ 1] 04 seminar macrobiotica - tratamentul cancerului.md | Tratamentul cancerului
     ### Tratamentul cancerului

[ 2] 04 seminar macrobiotica - tratamentul cancerului.md | Tratamentul cancerului
     Condimente: Gomashio preparat în proporție de 1 la 10 (susan) pentru cancerul Yin și 1 la 4 pentru cancerul Yang. Prunele Umeboshi convin amândurora. În cazurile Yang, se pot da aceste libertăți ocazional. Pentru cei cu ...

[ 3] 04 seminar macrobiotica - tratamentul cancerului.md | Studii de Caz și Statistici Internaționale
     1) Când bolnavul vrea să mănânce orice (lipsă de autodisciplină). De exemplu, dacă se spune că sosul Tamari trebuie luat în raport de 1/10 (sare/susan) și pacientul pune 1/6, preparatul va fi prea sărat. Profesorul KOHLE...

[ 4] 04 seminar macrobiotica -

## Inspecție liberă

Refolosește `traces[<întrebare>]` pentru a inspecta un obiect `RetrievalTrace` complet (`.semantic_hits`, `.lexical_hits`, `.fused`, `.reranked`, `.final`), sau rulează o întrebare nouă direct:

```python
trace = retrieve_with_trace("întrebarea mea ad-hoc")
show_trace(trace, top_n=20)
```